# Part 4: Regularization (Ridge, Lasso, ElasticNet) - Insurance Dataset

In this section, we explore regularization techniques: Ridge (L2), Lasso (L1), and ElasticNet (L1+L2) on medical insurance cost prediction. These methods prevent overfitting by penalizing large coefficients.

## Step 1: Import Necessary Libraries for Regularization

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

## Step 2: Load and Explore Insurance Dataset

**Dataset:** Medical Insurance Cost Prediction  
**Task:** Predict insurance charges based on health and demographic features

Features: age, sex, bmi, children, smoker, region  
Target: charges (medical insurance cost)

In [ ]:
# Load insurance dataset
# Option 1: From local file (if available)
df = pd.read_csv('../Machine_Learning-Data_Preprocssing_and_Feature_Engineering/insurance_aex5x.csv')

# Option 2: From Kaggle (requires kaggle API setup)
# from kaggle.api.kaggle_api_extended import KaggleApi
# api = KaggleApi()
# api.authenticate()
# api.dataset_download_files('mirichoi0219/insurance', path='./', unzip=True)
# df = pd.read_csv('insurance.csv')

print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())
print("\nDataset Info:")
print(df.info())
print("\nBasic Statistics:")
print(df.describe())

## Step 3: Data Preprocessing - Encoding & Train-Test Split

Convert categorical variables (sex, smoker, region) to numeric using one-hot encoding.  
This is mandatory before training regularized models.

In [ ]:
# One-hot encode categorical variables
df_encoded = pd.get_dummies(df, drop_first=True)

# Separate features and target
X = df_encoded.drop('charges', axis=1)
y = df_encoded['charges']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Feature Names:", list(X.columns))
print(f"\nTraining Set Size: {X_train.shape}")
print(f"Test Set Size: {X_test.shape}")
print(f"\nTarget Variable Statistics:")
print(f"Train - Mean: ${y_train.mean():.2f}, Std: ${y_train.std():.2f}")
print(f"Test  - Mean: ${y_test.mean():.2f}, Std: ${y_test.std():.2f}")

## Step 4: Feature Scaling (MANDATORY for Regularization)

⚠️ **CRITICAL:** Regularization penalties depend on feature magnitude.  
Without scaling, features with larger values will be penalized more heavily.

StandardScaler transforms features to mean=0, std=1.

In [ ]:
# Scale features using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Verify scaling: mean~0, std~1
print("Scaled Training Data Statistics:")
print(f"Mean: {X_train_scaled.mean(axis=0).round(4)}")
print(f"Std:  {X_train_scaled.std(axis=0).round(4)}")
print("\nFeature scaling completed - ready for regularized models!")

## Step 5: Baseline Model - Linear Regression (No Regularization)

Train a baseline linear regression model to use as comparison for regularized models.

In [ ]:
# Train baseline linear regression
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

# Predictions
y_pred_train_lr = lr.predict(X_train_scaled)
y_pred_test_lr = lr.predict(X_test_scaled)

# Evaluation
train_r2_lr = r2_score(y_train, y_pred_train_lr)
test_r2_lr = r2_score(y_test, y_pred_test_lr)
train_mse_lr = mean_squared_error(y_train, y_pred_train_lr)
test_mse_lr = mean_squared_error(y_test, y_pred_test_lr)
test_rmse_lr = np.sqrt(test_mse_lr)

print("LINEAR REGRESSION (Baseline - No Regularization)")
print("=" * 50)
print(f"Train R² Score: {train_r2_lr:.4f}")
print(f"Test R² Score:  {test_r2_lr:.4f}")
print(f"Train MSE: ${train_mse_lr:,.2f}")
print(f"Test MSE:  ${test_mse_lr:,.2f}")
print(f"Test RMSE: ${test_rmse_lr:,.2f}")
print(f"\nCoefficients (Top 5 by absolute value):")
coef_df_lr = pd.DataFrame({'Feature': X.columns, 'Coefficient': lr.coef_})
print(coef_df_lr.reindex(coef_df_lr['Coefficient'].abs().argsort()[-5:]))

## Step 6: Ridge Regression (L2 Regularization) - Manual Alpha Tuning

**Ridge (L2):** Penalty = α × (sum of squared coefficients)  
- All coefficients shrink towards zero (never exactly zero)  
- Better when all features are important  
- Reduces coefficient magnitude, especially for correlated features

In [ ]:
# Test different alpha values for Ridge manually
alphas = [0.001, 0.01, 0.1, 1, 10, 100, 1000]
ridge_results = []

print("RIDGE REGRESSION - Effect of Alpha (α)")
print("=" * 80)
print(f"{'Alpha':<10} {'Train R²':<12} {'Test R²':<12} {'Train MSE':<15} {'Test MSE':<15}")
print("-" * 80)

for alpha in alphas:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train_scaled, y_train)
    
    y_pred_train = ridge.predict(X_train_scaled)
    y_pred_test = ridge.predict(X_test_scaled)
    
    train_r2 = r2_score(y_train, y_pred_train)
    test_r2 = r2_score(y_test, y_pred_test)
    train_mse = mean_squared_error(y_train, y_pred_train)
    test_mse = mean_squared_error(y_test, y_pred_test)
    
    ridge_results.append({
        'Alpha': alpha,
        'Train_R2': train_r2,
        'Test_R2': test_r2,
        'Train_MSE': train_mse,
        'Test_MSE': test_mse
    })
    
    print(f"{alpha:<10.3f} {train_r2:<12.4f} {test_r2:<12.4f} {train_mse:<15,.0f} {test_mse:<15,.0f}")

ridge_results_df = pd.DataFrame(ridge_results)
best_alpha_ridge = ridge_results_df.loc[ridge_results_df['Test_R2'].idxmax(), 'Alpha']
print(f"\n✓ Best Alpha for Ridge: {best_alpha_ridge}")

## Step 7: Lasso Regression (L1 Regularization) - Manual Alpha Tuning

**Lasso (L1):** Penalty = α × (sum of absolute coefficient values)  
- Coefficients can shrink to EXACTLY zero (feature selection!)  
- Identifies less important features automatically  
- Better when you suspect many features are irrelevant

In [ ]:
# Test different alpha values for Lasso manually
lasso_results = []

print("LASSO REGRESSION - Effect of Alpha (α)")
print("=" * 100)
print(f"{'Alpha':<10} {'Train R²':<12} {'Test R²':<12} {'Train MSE':<15} {'Test MSE':<15} {'Zero Coef':<12}")
print("-" * 100)

for alpha in alphas:
    lasso = Lasso(alpha=alpha, max_iter=10000)
    lasso.fit(X_train_scaled, y_train)
    
    y_pred_train = lasso.predict(X_train_scaled)
    y_pred_test = lasso.predict(X_test_scaled)
    
    train_r2 = r2_score(y_train, y_pred_train)
    test_r2 = r2_score(y_test, y_pred_test)
    train_mse = mean_squared_error(y_train, y_pred_train)
    test_mse = mean_squared_error(y_test, y_pred_test)
    zero_coefs = (lasso.coef_ == 0).sum()  # Count features eliminated
    
    lasso_results.append({
        'Alpha': alpha,
        'Train_R2': train_r2,
        'Test_R2': test_r2,
        'Train_MSE': train_mse,
        'Test_MSE': test_mse,
        'Zero_Coefficients': zero_coefs
    })
    
    print(f"{alpha:<10.3f} {train_r2:<12.4f} {test_r2:<12.4f} {train_mse:<15,.0f} {test_mse:<15,.0f} {zero_coefs:<12}")

lasso_results_df = pd.DataFrame(lasso_results)
best_alpha_lasso = lasso_results_df.loc[lasso_results_df['Test_R2'].idxmax(), 'Alpha']
print(f"\n✓ Best Alpha for Lasso: {best_alpha_lasso}")
print(f"  This model eliminates {lasso_results_df.loc[lasso_results_df['Alpha']==best_alpha_lasso, 'Zero_Coefficients'].values[0]} features")

## Step 8: ElasticNet Regression (L1 + L2) - Manual Parameter Tuning

**ElasticNet:** Penalty = α × [l1_ratio × |coef| + (1-l1_ratio) × coef²]  
- Combines benefits of Ridge (L2) and Lasso (L1)  
- l1_ratio=0 → Ridge (all L2)  
- l1_ratio=1 → Lasso (all L1)  
- l1_ratio=0.5 → Balanced combination  
- Best when you want both feature selection AND coefficient shrinkage

In [ ]:
# Test ElasticNet with different alpha and l1_ratio combinations
elasticnet_results = []

print("ELASTICNET REGRESSION - Effect of Alpha and L1 Ratio")
print("=" * 110)
print(f"{'Alpha':<10} {'L1_Ratio':<12} {'Train R²':<12} {'Test R²':<12} {'Train MSE':<15} {'Test MSE':<15}")
print("-" * 110)

l1_ratios = [0, 0.5, 1.0]  # 0=Ridge, 0.5=Mixed, 1=Lasso

for alpha in [0.001, 0.01, 0.1, 1, 10, 100]:
    for l1_ratio in l1_ratios:
        en = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=10000)
        en.fit(X_train_scaled, y_train)
        
        y_pred_train = en.predict(X_train_scaled)
        y_pred_test = en.predict(X_test_scaled)
        
        train_r2 = r2_score(y_train, y_pred_train)
        test_r2 = r2_score(y_test, y_pred_test)
        train_mse = mean_squared_error(y_train, y_pred_train)
        test_mse = mean_squared_error(y_test, y_pred_test)
        
        elasticnet_results.append({
            'Alpha': alpha,
            'L1_Ratio': l1_ratio,
            'Train_R2': train_r2,
            'Test_R2': test_r2,
            'Train_MSE': train_mse,
            'Test_MSE': test_mse
        })
        
        l1_label = 'Ridge' if l1_ratio == 0 else ('Lasso' if l1_ratio == 1.0 else 'Mixed')
        print(f"{alpha:<10.3f} {l1_ratio:<12.1f} {train_r2:<12.4f} {test_r2:<12.4f} {train_mse:<15,.0f} {test_mse:<15,.0f}  ({l1_label})")

elasticnet_results_df = pd.DataFrame(elasticnet_results)
best_en = elasticnet_results_df.loc[elasticnet_results_df['Test_R2'].idxmax()]
print(f"\n✓ Best ElasticNet Parameters:")
print(f"  Alpha: {best_en['Alpha']}, L1_Ratio: {best_en['L1_Ratio']}")
print(f"  Test R²: {best_en['Test_R2']:.4f}")

## Step 9: GridSearchCV - Exhaustive Ridge Hyperparameter Optimization

Use GridSearchCV to systematically find the best alpha value for Ridge regression.  
This tests all combinations with 5-fold cross-validation.

In [ ]:
# GridSearchCV for Ridge Regression
ridge_param_grid = {
    'alpha': np.logspace(-3, 5, 20)  # 20 alpha values from 0.001 to 100000
}

ridge_gs = GridSearchCV(Ridge(), ridge_param_grid, cv=5, scoring='r2', n_jobs=-1)
ridge_gs.fit(X_train_scaled, y_train)

print("GRIDSEARCHCV - RIDGE REGRESSION")
print("=" * 70)
print(f"Best Alpha: {ridge_gs.best_params_['alpha']:.4f}")
print(f"Best CV R² Score: {ridge_gs.best_score_:.4f}")
print(f"Total Combinations Tested: {len(ridge_gs.cv_results_['params'])}")
print(f"Total Model Fits (CV): {len(ridge_gs.cv_results_['params']) * 5} (20 alphas × 5-fold)")

# Evaluate best model on test set
ridge_best = ridge_gs.best_estimator_
y_pred_ridge = ridge_best.predict(X_test_scaled)
ridge_test_r2 = r2_score(y_test, y_pred_ridge)
ridge_test_mse = mean_squared_error(y_test, y_pred_ridge)
ridge_test_rmse = np.sqrt(ridge_test_mse)

print(f"\nTest Set Performance (Best Ridge):")
print(f"Test R² Score: {ridge_test_r2:.4f}")
print(f"Test MSE: ${ridge_test_mse:,.2f}")
print(f"Test RMSE: ${ridge_test_rmse:,.2f}")

## Step 10: GridSearchCV - Exhaustive Lasso Hyperparameter Optimization

Find optimal alpha for Lasso using GridSearchCV with cross-validation.

In [ ]:
# GridSearchCV for Lasso Regression
lasso_param_grid = {
    'alpha': np.logspace(-4, 3, 20),  # 20 alpha values from 0.0001 to 1000
    'max_iter': [5000]
}

lasso_gs = GridSearchCV(Lasso(), lasso_param_grid, cv=5, scoring='r2', n_jobs=-1)
lasso_gs.fit(X_train_scaled, y_train)

print("GRIDSEARCHCV - LASSO REGRESSION")
print("=" * 70)
print(f"Best Alpha: {lasso_gs.best_params_['alpha']:.4f}")
print(f"Best CV R² Score: {lasso_gs.best_score_:.4f}")
print(f"Total Combinations Tested: {len(lasso_gs.cv_results_['params'])}")
print(f"Total Model Fits (CV): {len(lasso_gs.cv_results_['params']) * 5}")

# Evaluate best model on test set
lasso_best = lasso_gs.best_estimator_
y_pred_lasso = lasso_best.predict(X_test_scaled)
lasso_test_r2 = r2_score(y_test, y_pred_lasso)
lasso_test_mse = mean_squared_error(y_test, y_pred_lasso)
lasso_test_rmse = np.sqrt(lasso_test_mse)
lasso_zero_coefs = (lasso_best.coef_ == 0).sum()

print(f"\nTest Set Performance (Best Lasso):")
print(f"Test R² Score: {lasso_test_r2:.4f}")
print(f"Test MSE: ${lasso_test_mse:,.2f}")
print(f"Test RMSE: ${lasso_test_rmse:,.2f}")
print(f"Features Eliminated (Zero Coefficients): {lasso_zero_coefs}/{len(X.columns)}")

## Step 11: GridSearchCV - ElasticNet Parameter Grid Search

Optimize ElasticNet by tuning both alpha and l1_ratio (2D parameter grid).

In [ ]:
# GridSearchCV for ElasticNet (2D parameter grid)
elasticnet_param_grid = {
    'alpha': np.logspace(-3, 3, 10),  # 10 alpha values
    'l1_ratio': [0, 0.25, 0.5, 0.75, 1.0],  # 5 l1_ratio values
    'max_iter': [5000]
}

elasticnet_gs = GridSearchCV(ElasticNet(), elasticnet_param_grid, cv=5, scoring='r2', n_jobs=-1)
elasticnet_gs.fit(X_train_scaled, y_train)

print("GRIDSEARCHCV - ELASTICNET REGRESSION (2D Grid Search)")
print("=" * 80)
print(f"Best Alpha: {elasticnet_gs.best_params_['alpha']:.4f}")
print(f"Best L1 Ratio: {elasticnet_gs.best_params_['l1_ratio']:.2f}")
print(f"Best CV R² Score: {elasticnet_gs.best_score_:.4f}")
print(f"Total Combinations Tested: {len(elasticnet_gs.cv_results_['params'])}")
print(f"Total Model Fits (CV): {len(elasticnet_gs.cv_results_['params']) * 5}")

# Evaluate best model on test set
elasticnet_best = elasticnet_gs.best_estimator_
y_pred_elasticnet = elasticnet_best.predict(X_test_scaled)
elasticnet_test_r2 = r2_score(y_test, y_pred_elasticnet)
elasticnet_test_mse = mean_squared_error(y_test, y_pred_elasticnet)
elasticnet_test_rmse = np.sqrt(elasticnet_test_mse)

print(f"\nTest Set Performance (Best ElasticNet):")
print(f"Test R² Score: {elasticnet_test_r2:.4f}")
print(f"Test MSE: ${elasticnet_test_mse:,.2f}")
print(f"Test RMSE: ${elasticnet_test_rmse:,.2f}")

## Step 12: Model Comparison - Coefficient Analysis

Compare coefficients across all models to understand regularization effects.

In [ ]:
# Coefficient comparison across all models
coef_comparison = pd.DataFrame({
    'Feature': X.columns,
    'Linear': lr.coef_,
    'Ridge': ridge_best.coef_,
    'Lasso': lasso_best.coef_,
    'ElasticNet': elasticnet_best.coef_
})

print("COEFFICIENT COMPARISON ACROSS MODELS")
print("=" * 100)
print(coef_comparison.to_string())

print("\n\nKEY OBSERVATIONS:")
print("-" * 100)
print(f"Ridge:      All coefficients shrunk but preserved (max: {ridge_best.coef_.max():.4f})")
print(f"Lasso:      {(lasso_best.coef_ == 0).sum()} features eliminated to zero")
print(f"ElasticNet: {(elasticnet_best.coef_ == 0).sum()} features eliminated, balanced shrinkage")
print(f"Linear:     Baseline with no regularization")

# Most important features
print("\n\nTOP 5 IMPORTANT FEATURES (by Linear Regression):")
print("-" * 100)
top_features = coef_comparison.reindex(coef_comparison['Linear'].abs().argsort()[-5:])
print(top_features.to_string())

## Step 13: Visualization - Alpha Effects on Model Performance

Visualize how changes in alpha affect Ridge and Lasso performance.

In [ ]:
# Visualize alpha effects on Ridge and Lasso
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Regularization Effects: Alpha Impact on Model Performance', fontsize=16, fontweight='bold')

# Ridge R² vs Alpha
ax1 = axes[0, 0]
ax1.semilogx(ridge_results_df['Alpha'], ridge_results_df['Train_R2'], 'o-', label='Train R²', linewidth=2, markersize=6)
ax1.semilogx(ridge_results_df['Alpha'], ridge_results_df['Test_R2'], 's-', label='Test R²', linewidth=2, markersize=6)
ax1.axvline(best_alpha_ridge, color='red', linestyle='--', alpha=0.7, label=f'Best α={best_alpha_ridge:.3f}')
ax1.set_xlabel('Alpha (α)', fontsize=11, fontweight='bold')
ax1.set_ylabel('R² Score', fontsize=11, fontweight='bold')
ax1.set_title('Ridge Regression: R² vs Alpha', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Ridge MSE vs Alpha
ax2 = axes[0, 1]
ax2.loglog(ridge_results_df['Alpha'], ridge_results_df['Train_MSE'], 'o-', label='Train MSE', linewidth=2, markersize=6)
ax2.loglog(ridge_results_df['Alpha'], ridge_results_df['Test_MSE'], 's-', label='Test MSE', linewidth=2, markersize=6)
ax2.axvline(best_alpha_ridge, color='red', linestyle='--', alpha=0.7, label=f'Best α={best_alpha_ridge:.3f}')
ax2.set_xlabel('Alpha (α)', fontsize=11, fontweight='bold')
ax2.set_ylabel('MSE', fontsize=11, fontweight='bold')
ax2.set_title('Ridge Regression: MSE vs Alpha', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

# Lasso R² vs Alpha
ax3 = axes[1, 0]
ax3.semilogx(lasso_results_df['Alpha'], lasso_results_df['Train_R2'], 'o-', label='Train R²', linewidth=2, markersize=6)
ax3.semilogx(lasso_results_df['Alpha'], lasso_results_df['Test_R2'], 's-', label='Test R²', linewidth=2, markersize=6)
ax3.axvline(best_alpha_lasso, color='red', linestyle='--', alpha=0.7, label=f'Best α={best_alpha_lasso:.3f}')
ax3.set_xlabel('Alpha (α)', fontsize=11, fontweight='bold')
ax3.set_ylabel('R² Score', fontsize=11, fontweight='bold')
ax3.set_title('Lasso Regression: R² vs Alpha', fontsize=12, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3)

# Lasso - Zero Coefficients vs Alpha
ax4 = axes[1, 1]
ax4.semilogx(lasso_results_df['Alpha'], lasso_results_df['Zero_Coefficients'], 'D-', color='orange', linewidth=2, markersize=8)
ax4.axvline(best_alpha_lasso, color='red', linestyle='--', alpha=0.7, label=f'Best α={best_alpha_lasso:.3f}')
ax4.set_xlabel('Alpha (α)', fontsize=11, fontweight='bold')
ax4.set_ylabel('Number of Zero Coefficients', fontsize=11, fontweight='bold')
ax4.set_title('Lasso Regression: Feature Elimination vs Alpha', fontsize=12, fontweight='bold')
ax4.set_ylim([-0.5, len(X.columns) + 0.5])
ax4.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("✓ Visualization complete: Observe how alpha affects model performance and feature selection")

## Step 14: Final Model Comparison Dashboard

Compare all models using comprehensive evaluation metrics and visualizations.

In [ ]:
# Comprehensive model comparison
print("=" * 110)
print("COMPREHENSIVE MODEL COMPARISON - INSURANCE DATASET")
print("=" * 110)

model_comparison = pd.DataFrame({
    'Model': ['Linear (Baseline)', 'Ridge (GridSearchCV)', 'Lasso (GridSearchCV)', 'ElasticNet (GridSearchCV)'],
    'Test_R2': [test_r2_lr, ridge_test_r2, lasso_test_r2, elasticnet_test_r2],
    'Test_MSE': [test_mse_lr, ridge_test_mse, lasso_test_mse, elasticnet_test_mse],
    'Test_RMSE': [test_rmse_lr, ridge_test_rmse, lasso_test_rmse, elasticnet_test_rmse]
})

print("\nTest Set Performance Metrics:")
print(model_comparison.to_string(index=False))

# Calculate improvements
best_model_idx = model_comparison['Test_R2'].idxmax()
best_model_name = model_comparison.loc[best_model_idx, 'Model']
best_r2 = model_comparison.loc[best_model_idx, 'Test_R2']
baseline_r2 = model_comparison.loc[0, 'Test_R2']

print(f"\n{'✓ BEST MODEL: ' + best_model_name}")
print(f"  Improvement over baseline: {((best_r2 - baseline_r2) / baseline_r2 * 100):.2f}%")

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Model Comparison Dashboard - Insurance Cost Prediction', fontsize=16, fontweight='bold')

models = model_comparison['Model'].str.replace(' (GridSearchCV)', '')
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

# R² Comparison
ax1 = axes[0, 0]
bars1 = ax1.bar(models, model_comparison['Test_R2'], color=colors, edgecolor='black', linewidth=1.5)
ax1.set_ylabel('R² Score', fontsize=11, fontweight='bold')
ax1.set_title('R² Score Comparison (Higher is Better)', fontsize=12, fontweight='bold')
ax1.set_ylim([0, 1])
ax1.grid(True, alpha=0.3, axis='y')
for i, (bar, val) in enumerate(zip(bars1, model_comparison['Test_R2'])):
    ax1.text(bar.get_x() + bar.get_width()/2, val + 0.02, f'{val:.4f}', 
             ha='center', va='bottom', fontweight='bold', fontsize=10)

# MSE Comparison
ax2 = axes[0, 1]
bars2 = ax2.bar(models, model_comparison['Test_MSE'], color=colors, edgecolor='black', linewidth=1.5)
ax2.set_ylabel('Mean Squared Error (MSE)', fontsize=11, fontweight='bold')
ax2.set_title('MSE Comparison (Lower is Better)', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')
for i, (bar, val) in enumerate(zip(bars2, model_comparison['Test_MSE'])):
    ax2.text(bar.get_x() + bar.get_width()/2, val + 500000, f'${val:,.0f}', 
             ha='center', va='bottom', fontweight='bold', fontsize=9)

# RMSE Comparison
ax3 = axes[1, 0]
bars3 = ax3.bar(models, model_comparison['Test_RMSE'], color=colors, edgecolor='black', linewidth=1.5)
ax3.set_ylabel('Root Mean Squared Error (RMSE)', fontsize=11, fontweight='bold')
ax3.set_title('RMSE Comparison - Prediction Error in $ (Lower is Better)', fontsize=12, fontweight='bold')
ax3.grid(True, alpha=0.3, axis='y')
for i, (bar, val) in enumerate(zip(bars3, model_comparison['Test_RMSE'])):
    ax3.text(bar.get_x() + bar.get_width()/2, val + 500, f'${val:,.0f}', 
             ha='center', va='bottom', fontweight='bold', fontsize=9)

# Coefficient distribution comparison
ax4 = axes[1, 1]
coef_stats = pd.DataFrame({
    'Linear': [lr.coef_.mean(), lr.coef_.std(), np.abs(lr.coef_).max()],
    'Ridge': [ridge_best.coef_.mean(), ridge_best.coef_.std(), np.abs(ridge_best.coef_).max()],
    'Lasso': [lasso_best.coef_.mean(), lasso_best.coef_.std(), np.abs(lasso_best.coef_).max()],
    'ElasticNet': [elasticnet_best.coef_.mean(), elasticnet_best.coef_.std(), np.abs(elasticnet_best.coef_).max()]
}, index=['Mean', 'Std Dev', 'Max Abs Value'])

x_pos = np.arange(len(coef_stats.columns))
width = 0.25
for i, metric in enumerate(coef_stats.index):
    ax4.bar(x_pos + i*width, coef_stats.loc[metric], width, label=metric, edgecolor='black', linewidth=1)

ax4.set_ylabel('Coefficient Statistics', fontsize=11, fontweight='bold')
ax4.set_title('Coefficient Distribution Across Models', fontsize=12, fontweight='bold')
ax4.set_xticks(x_pos + width)
ax4.set_xticklabels(coef_stats.columns)
ax4.legend(fontsize=10)
ax4.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\n" + "=" * 110)

# Summary: Regularization Techniques (Ridge vs Lasso vs ElasticNet)

## Key Learnings

### 1. **What is Regularization?**
Regularization adds a penalty term to prevent overfitting by controlling coefficient magnitude:
- **Unregularized Cost** = MSE
- **Regularized Cost** = MSE + α × Penalty(coefficients)

### 2. **Ridge Regression (L2 Regularization)**
- **Penalty:** α × (sum of squared coefficients)
- **Effect:** All coefficients shrink together but never become exactly zero
- **Best for:** When all features are believed to be important
- **Formula:** minimize: MSE + α × Σ(β²)

### 3. **Lasso Regression (L1 Regularization)**
- **Penalty:** α × (sum of absolute coefficients)
- **Effect:** Coefficients can shrink to EXACTLY zero → automatic feature selection
- **Best for:** High-dimensional data where many features might be irrelevant
- **Formula:** minimize: MSE + α × Σ|β|

### 4. **ElasticNet Regression (L1 + L2)**
- **Penalty:** α × [l1_ratio × Σ|β| + (1-l1_ratio) × Σ(β²)]
- **Effect:** Combines benefits of both Ridge and Lasso
- **Best for:** When you want both shrinkage AND feature selection
- **l1_ratio:** 
  - 0 = Ridge (pure L2)
  - 1 = Lasso (pure L1)
  - 0.5 = Balanced

### 5. **Impact of Alpha (λ)**
- **Alpha too small (0):** No regularization → may overfit
- **Alpha too large:** Excessive shrinkage → underfitting
- **Optimal Alpha:** Found via cross-validation (GridSearchCV)

### 6. **When to Use Each**
| Scenario | Best Model |
|----------|-----------|
| Many features, suspect many are irrelevant | **Lasso** |
| All features correlated & important | **Ridge** |
| High-dim with some irrelevant features | **ElasticNet** |
| Need coefficient interpretation | **Ridge/Lasso** |
| Want stability + feature selection | **ElasticNet** |

### 7. **Important Implementation Rules**
⚠️ **ALWAYS SCALE FEATURES BEFORE REGULARIZATION**
- Regularization penalties depend on feature magnitude
- StandardScaler (mean=0, std=1) is essential
- Otherwise, features with larger values get penalized more

### 8. **Hyperparameter Tuning**
- **GridSearchCV:** Test all combinations exhaustively
- **Ridge:** Tune α (1 parameter)
- **Lasso:** Tune α (1 parameter)
- **ElasticNet:** Tune α + l1_ratio (2D grid)

### 9. **Model Selection Criteria**
1. Start with GridSearchCV for both Ridge and Lasso
2. Compare test R² scores
3. Check coefficient interpretability
4. Consider computational cost
5. ElasticNet if hybrid approach needed

### 10. **Practical Tips**
- Use LogScale for alpha: np.logspace(-3, 3) covers wide range
- Always validate with test set (not just CV scores)
- Plot alpha vs performance to understand regularization effect
- Feature elimination (Lasso) indicates irrelevant features
- Coefficient magnitude reduction indicates feature importance
